In [ ]:
basedir = r'E:\University of Michigan Dropbox\Jiahui Chen'
basedir = r'C:\Users\jiahuic\University of Michigan Dropbox\Jiahui Chen'

In [ ]:
# territory finding test
import geopandas as gpd
from shapely.geometry import Point
# Load the shapefile
shapefile_path = basedir + r"\\PVEV\UtilityTerritory\Electric_Retail_Service_Territories.shp"  # Replace with the actual path to your shapefile
gdf = gpd.read_file(shapefile_path)

# Function to find the utility name based on longitude and latitude
def find_utility_name(longitude, latitude):
    """
    Finds the utility name based on the given longitude and latitude.

    Parameters:
        longitude (float): Longitude of the point.
        latitude (float): Latitude of the point.

    Returns:
        str: Name of the utility, or None if no match is found.
    """
    # Create a Point object for the given coordinates
    point = Point(longitude, latitude)
    lst = []
    # Check which polygon contains the point
    for _, row in gdf.iterrows():
        if row['geometry'].contains(point):
            # Assuming the utility name is stored in a column named 'NAME'
            lst.append(row.get('NAME', 'Unknown'))

    return lst

# Example usage
if __name__ == "__main__":
    # Input coordinates
    longitude = -83.0458  # Example longitude
    latitude = 42.3314    # Example latitude

    # Find the utility name
    utility_name = find_utility_name(longitude, latitude)
    if utility_name:
        print(utility_name)
    else:
        print("No utility found for the given location.")

In [ ]:
import geopandas as gpd
import pandas as pd
import os
from shapely.geometry import Point
# Load the shapefile
shapefile_path = basedir + r"\\PVEV\UtilityTerritory\Electric_Retail_Service_Territories.shp"  # Replace with the actual path to your shapefile
urdb_df = pd.read_csv(basedir + r'\\PVEV\julia\filtered_usurdb.csv')
gdf = gpd.read_file(shapefile_path)


In [ ]:
import numpy as np
# Initialize a new column for matched utility names
# Add a column to urdb_df with stripped and lowercase strings
urdb_df['utility_lower'] = urdb_df['utility'].str.strip().str.lower()

# Add a column to gdf with stripped and lowercase strings
gdf['NAME_lower'] = gdf['NAME'].str.strip().str.lower()

# Extract all unique utility names from the shapefile
utility_names = gdf['NAME_lower'].unique()  # Replace 'NAME' with the actual column name if different
print(utility_names)

In [ ]:
import pandas as pd
ctdf = pd.read_csv(basedir + r'\\Ford_CC\2020_Gaz_counties_national_0728_tempreg.csv')
import numpy as np

for i in range(4):
    ctdf[f'util_{i}'] = np.zeros(len(ctdf))

In [ ]:
# place-utility
for i in range(len(ctdf['INTPTLONG'])):
    util_list = find_utility_name(ctdf.loc[i, 'INTPTLONG'], ctdf.loc[i, 'INTPTLAT'])
    for ii in range(len(util_list)):
        if ii < 4:
            ctdf.loc[i, f'util_{ii}'] = util_list[ii]

In [ ]:
lst = []
for i in range(4):
    lst= lst +list(ctdf['util_' + str(i)].unique())
print(lst)
print(len(lst))
not_in_df_sales = [name for name in lst if name.strip().lower() not in name_sales_lower]
print(not_in_df_sales)
print(len(not_in_df_sales))

In [ ]:
df_sales = pd.read_excel(basedir + r'\PVEV\861\Sales_Ult_Cust_2023_processed.xlsx')
df_sales = df_sales.iloc[:-1]
name_cs = lst
name_cs = list(set(name_cs)) # remove duplicates
name_sales = df_sales['Utility Name'].to_list()
name_sales = list(set(name_sales)) # remove duplicates
for i in range(len(name_cs)):
    name_cs[i] = name_cs[i].strip().lower()

# Find elements in name_cs that are not in name_sales, case insensitive
name_sales_lower = [name.strip().lower() for name in name_sales]
not_in_name_sales = [name for name in name_cs if name not in name_sales_lower]
print(not_in_name_sales)
print(len(not_in_name_sales))


In [ ]:
# Initialize a new column for the utility with the highest sales
ctdf['util_maxsales'] = None

# Iterate through each row in df1
for index, row in ctdf.iterrows():
    # Extract the utilities for the current row
    utilities = [row[f'util_{i}'] for i in range(4)]
    
    # Filter df_sales for the utilities in the current row
    sales_data = df_sales[df_sales['Utility Name'].str.strip().str.lower().isin([str(util).strip().lower() for util in utilities])]
    # Remove rows where 'ResSales' is not a number
    sales_data = sales_data[pd.to_numeric(sales_data['ResSales'], errors='coerce').notna()]
    # Find the utility with the highest residential sales
    if not sales_data.empty:
        max_sales_utility = sales_data.loc[sales_data['ResSales'].idxmax(), 'Utility Name']
        ctdf.at[index, 'util_maxsales'] = max_sales_utility

In [ ]:
# Initialize lists to store the chosen utility and the reason
chosen_utilities = []
reasons = []

# Iterate through each tempreg region
for regi in range(432):
    # Get the list of utilities for the current region
    utility_list = list(ctdf.loc[ctdf['tempreg'] == regi, 'util_maxsales'])
    print(utility_list)
    
    # Count the occurrences of each utility in the list
    utility_counts = pd.Series(utility_list).value_counts()

    # Find the utility with the highest count
    max_count_utilities = utility_counts[utility_counts == utility_counts.max()].index.tolist()

    if len(max_count_utilities) == 1:
        # If there's a single utility with the highest count, choose it
        chosen_utilities.append(max_count_utilities[0])
        reasons.append("most count")
    else:
        # If there's a tie, find the one with the highest sales
        sales_data = df_sales[df_sales['Utility Name'].str.strip().str.lower().isin([util.strip().lower() for util in max_count_utilities])]
        sales_data = sales_data[pd.to_numeric(sales_data['ResSales'], errors='coerce').notna()]
        if not sales_data.empty:
            max_sales_utility = sales_data.loc[sales_data['ResSales'].idxmax(), 'Utility Name']
            chosen_utilities.append(max_sales_utility)
            reasons.append("tie & most sales")
        else:
            # If no sales data is available, choose the first utility in the tie
            chosen_utilities.append(max_count_utilities[0])
            reasons.append("tie & no sales data")

# Map the chosen utilities and reasons back to the DataFrame
ctdf['chosen_utility'] = ctdf['tempreg'].map(dict(zip(range(432), chosen_utilities)))
ctdf['reason'] = ctdf['tempreg'].map(dict(zip(range(432), reasons)))

In [ ]:
# Add a new column to ctdf for the matched utility
ctdf['matched_utility'] = ctdf['chosen_utility'].str.strip().str.lower().map(
    filtered_usurdb.set_index('utility')['utility'].str.strip().str.lower()
)

# Fill in 'no match' for rows where no match is found
ctdf['matched_utility'].fillna('no match', inplace=True)

In [ ]:
# Group by 'tempreg' and collect unique utility names
tempreg_utilities = ctdf.groupby('tempreg')['util_maxsales'].unique()

# Find tempregs with no matches (i.e., all utility names are None)
no_match_tempregs = [tempreg for tempreg, utilities in tempreg_utilities.items() if all(utility is None for utility in utilities)]

# Print the results
print("Utilities for each tempreg:")
# for tempreg, utilities in tempreg_utilities.items():
    # print(f"Tempreg {tempreg}: {utilities}")

print("\nTempregs with no matches:")
print(no_match_tempregs)

In [ ]:
# Update 'util_maxsales' for specific tempreg values
ctdf.loc[ctdf['tempreg'].isin([23, 24, 25]), 'util_maxsales'] = 'DTE Electric Company'
ctdf.loc[ctdf['tempreg'].isin([148, 149]), 'util_maxsales'] = 'PacifiCorp'
# ctdf.loc[ctdf['tempreg'].isin([148, 149]), 'util_maxsales'] = 'PacifiCorp (Wyoming)'
ctdf.loc[ctdf['tempreg'] == 158, 'util_maxsales'] = 'PacifiCorp'
# ctdf.loc[ctdf['tempreg'] == 158, 'util_maxsales'] = 'PacifiCorp (Utah)'
ctdf.loc[ctdf['tempreg'] == 217, 'util_maxsales'] = 'Northern States Power Co - Minnesota'
ctdf.loc[ctdf['tempreg'] == 344, 'util_maxsales'] = 'Wisconsin Public Service Corp'
ctdf.loc[ctdf['tempreg'] == 401, 'util_maxsales'] = 'Florida Power & Light Co'

In [ ]:
cty_utility_lst[0][1]

In [ ]:
ctdf.to_csv(basedir + r'\\PVEV\utility_matching\2020_Gaz_counties_national_0728_tempreg_withutility.csv')

In [ ]:
# Load the filtered_usurdb.csv file
filtered_usurdb_path = basedir + r'\\PVEV\\julia\\filtered_usurdb.csv'
filtered_usurdb = pd.read_csv(filtered_usurdb_path)
util_usurdb = filtered_usurdb['utility'].unique()
# Convert the utility names to lowercase and strip whitespace for matching

# Convert util_usurdb to a set of lowercase utility names for efficient matching
util_usurdb_set = set(map(str.lower, util_usurdb))

# Map chosen_utility to the 'utility' column of filtered_usurdb
ctdf['matched_utility'] = ctdf['chosen_utility'].str.lower().apply(
	lambda x: x if x in util_usurdb_set else None
)

# Fill in 'no match' for rows where no match is found
ctdf['matched_utility'].fillna('no match', inplace=True)


In [ ]:
ctdf.to_csv(basedir + r'\\PVEV\utility_matching\2020_Gaz_counties_national_0728_tempreg_withutility_usurdb.csv')


fill in utility names in '2020_Gaz_counties_national_0728_tempreg_withutility_usurdb' manually if no match.
tempreg 13: duke energy florida, not available, changed to Florida Power & Light Co.
27 Indiana: Northern Indiana Pub Serv Co, changed to Duke Energy Indiana Inc
Indiana Michigan Power Co (Indiana)
49 Cleveland Electric Illum Co, ohio edison co
54 Dayton Power & Light Co, ohio edison co

if the RGU has both a company and its subsidiaries, fill in subs 
120, 121 maine: versant, central maine power co
124 coop, avista corp
199 otter tail power co
200 NorthWestern Energy
204 high west energy, inc
209, 216 Omaha Public Power District, no available, available too small
TX, KS changed to utility w/ biggest sales in state, too many coop
387 mississippi power co
389 

Match sales, revenues and count records in 861\Sales_Ult_Cust_2023_processed.csv to ctdf

In [ ]:
# 
# Load the sales data
ctdf = pd.read_csv(basedir + r'\\PVEV\utility_matching\2020_Gaz_counties_national_0728_tempreg_withutility_usurdb_filled.csv')
sales_data_path = basedir + r'\\PVEV\\861\\Sales_Ult_Cust_2023_processed.csv'
sales_data = pd.read_csv(sales_data_path)

# Ensure case-insensitive matching by converting to lowercase and stripping whitespace
sales_data['Utility Name'] = sales_data['Utility Name'].str.strip().str.lower()
ctdf['matched_utility'] = ctdf['matched_utility'].str.strip().str.lower()

# Initialize new columns in ctdf for residential sales, revenue, and count
ctdf['ResSales'] = None
ctdf['ResRev'] = None
ctdf['ResCount'] = None

# Map the sales data to ctdf based on the matched_utility column
for index, row in ctdf.iterrows():
    matched_utility = row['matched_utility']
    if matched_utility != 'no match':
        matching_sales = sales_data[sales_data['Utility Name'] == matched_utility]
        if not matching_sales.empty:
            ctdf.at[index, 'ResSales'] = matching_sales['ResSales'].values[0]
            ctdf.at[index, 'ResRev'] = matching_sales['ResRev'].values[0]
            ctdf.at[index, 'ResCount'] = matching_sales['ResCount'].values[0]
ctdf.to_csv(basedir + r'\\PVEV\utility_matching\2020_Gaz_counties_national_0728_tempreg_withutility_usurdb_filled_sales.csv')

NE and TX changed to available utility w most customers 
TX: entergy available, as in NREL's REopt
TX: southwestern electric power co no TOU, changed to entergy
TX: riogrande, old
TN duck river no update, changed to the most sales
MT counties only small coop, change to max sales
FL, 400 no avail in urdb, change to florida power & light co.
ID 127 coop too small change to max sales
ID 130 coop too old, change to max sales
KS 255-259, 262 coop too small, too old, changed to Evergy (sales)
KY 37,41 coop too old, no tou, changed to max sales
KY 406-408 coop too old, too small
ME 121 coop too old small
MI 337-341 too old
MN 215 218 220 221 224 225 too old
MO coop too old
MS 385-388
MT old
ND all old, changed to northern states minesota, max sales
NE coop old, changed to max sales
NM 174 176 177
ohio dayton no tou to ohio max sales 
SD northwestern energy no tou, changed
TN Nashville to Middle Tennessee E M C, geographical
TN memphis to City of Memphis, Tennessee (Utility Company)
TN 1st and 3rd largest utilities
WV monongahela power co (west virginia) no tou
WY coop too small & old, change to max sale
The next cell redoes the sales mapping by chosen utility and state:

In [ ]:
# do a new sales data mapping based on the chosen_utility and state, fill in where the earlier mapping is blank
# Load the sales data
ctdf = pd.read_csv(basedir + r'\\PVEV\utility_matching\2020_Gaz_counties_national_0728_tempreg_withutility_usurdb_filled.csv')
sales_data_path = basedir + r'\\PVEV\\861\\Sales_Ult_Cust_2023_processed.csv'
sales_data = pd.read_csv(sales_data_path)

# Ensure case-insensitive matching by converting to lowercase and stripping whitespace
sales_data['Utility Name'] = sales_data['Utility Name'].str.strip().str.lower()
ctdf['matched_utility'] = ctdf['matched_utility'].str.strip().str.lower()

# Initialize new columns in ctdf for residential sales, revenue, and count
ctdf['ResSales'] = None
ctdf['ResRev'] = None
ctdf['ResCount'] = None

# Map the sales data to ctdf based on the chosen_utility column and match State using USPS
ctdf['USPS'] = ctdf['USPS'].str.strip().str.upper()  # Ensure State column is uppercase and stripped
sales_data['State'] = sales_data['State'].str.strip().str.upper()  # Ensure USPS column is uppercase and stripped

# Filter sales data to match both chosen_utility and State
for index, row in ctdf.iterrows():
    chosen_utility = row['chosen_utility'].strip().lower()
    state = row['USPS']
    matching_sales = sales_data[(sales_data['Utility Name'] == chosen_utility) & (sales_data['State'] == state)]
    if not matching_sales.empty:
        ctdf.at[index, 'ResSales'] = matching_sales['ResSales'].values[0]
        ctdf.at[index, 'ResRev'] = matching_sales['ResRev'].values[0]
        ctdf.at[index, 'ResCount'] = matching_sales['ResCount'].values[0]

ctdf_error = pd.read_csv(basedir + r'\\PVEV\utility_matching\2020_Gaz_counties_national_0728_tempreg_withutility_usurdb_filled_sales_filled.csv')
# Fill blank cells in ResSales, ResRev, and ResCount with values from ctdf_error
for index, row in ctdf.iterrows():
    if pd.isna(row['ResSales']) or pd.isna(row['ResRev']) or pd.isna(row['ResCount']):
        ctdf.at[index, 'ResSales'] = ctdf_error.at[index, 'ResSales']
        ctdf.at[index, 'ResRev'] = ctdf_error.at[index, 'ResRev']
        ctdf.at[index, 'ResCount'] = ctdf_error.at[index, 'ResCount']
        ctdf.at[index, 'matched_utility'] = ctdf_error.at[index, 'matched_utility']
# for index, row in ctdf.iterrows():
#     matched_utility = row['matched_utility']
#     if matched_utility != 'no match':
#         matching_sales = sales_data[sales_data['Utility Name'] == matched_utility]
#         if not matching_sales.empty:
#             ctdf.at[index, 'ResSales'] = matching_sales['ResSales'].values[0]
#             ctdf.at[index, 'ResRev'] = matching_sales['ResRev'].values[0]
#             ctdf.at[index, 'ResCount'] = matching_sales['ResCount'].values[0]
ctdf.to_csv(basedir + r'\\PVEV\utility_matching\tempreg_withutility_usurdb_filled_sales_state.csv')


In [ ]:
len(ctdf[ctdf['matched_utility'] == 'no match'])

In [ ]:
ctdf

In [ ]:
lst_error = []
for name in urdb_df['matched_utility'].unique():
    if name not in utility_names:
        lst_error.append(name)
print(lst_error)

Coverage calculation

In [ ]:
import pandas as pd
import os
import numpy as np
urdb_df = pd.read_csv(basedir + r'\\PVEV\julia\usurdb.csv')
ctdf = pd.read_csv(basedir + r'\\PVEV\utility_matching\tempreg_withutility_usurdb_filled_sales_state.csv')
# Convert the 'ResSales' column to numeric by removing commas and converting to float
ctdf['ResSales'] = ctdf['ResSales'].str.replace(',', '').astype(float)
sum(ctdf['ResSales'].unique())/10**6

In [ ]:
sales_data = pd.read_csv(basedir + r'\\PVEV\\861\\Sales_Ult_Cust_2023_processed.csv')
sales_data = sales_data[~sales_data['State'].isin(['AK', 'HI'])]
sales_data['ResSales'] = sales_data['ResSales'].replace('.', '0')
sales_data['ResSales'] = sales_data['ResSales'].str.replace(',', '').astype(float)
sales_data.loc[sales_data['Part'].isin(['A','B','D'])]
sum(sales_data['ResSales'])/10**6

In [ ]:
sum(sales_data.loc[sales_data['Utility Name']=='Adjustment 2023','ResSales'])/10**6

In [ ]:
sales_data = pd.read_csv(basedir + r'\\PVEV\\861\\Sales_Ult_Cust_2023_processed.csv')
sales_data = sales_data[~sales_data['State'].isin(['AK', 'HI'])]
sales_data['ResSales'] = sales_data['ResSales'].replace('.', '0')
sales_data['ResSales'] = sales_data['ResSales'].str.replace(',', '').astype(float)
sales_data1 = sales_data.loc[sales_data['Part'].isin(['A','B','D'])]
sales_data1 = sales_data1.loc[sales_data['Utility Name'].isin(ctdf['chosen_utility'].unique())]

sum(sales_data1['ResSales'])/10**6

In [ ]:
734.506943 / 1445.281356

RATE calculation
tempregi
-->  utility name
-->  urdb program list 
-->  urdb tou program list
-->  check locational specific
-->  2023 utility average comparison
-->  record name and label in ctdf, each update is a number in file name

In [ ]:
non_single = []

In [ ]:
non_single

In [ ]:
tempregi

In [ ]:
import pandas as pd
import os
import numpy as np
import webbrowser
# auto process rgus with single rates, find non-single rate RGUs, manually process them later
urdb_df = pd.read_csv(basedir + r'\\PVEV\julia\residential_usurdb.csv')
ctdf = pd.read_csv(basedir + r'\\PVEV\utility_matching\tempreg_withutility_usurdb_filled_sales_state.csv')
for tempregi in range(10,432):
    ctdf1 = ctdf.loc[ctdf['tempreg'] == tempregi]
    util_name = ctdf1['matched_utility'].str.lower().unique()[0]
    urdb_df_1 = urdb_df.loc[urdb_df['utility'].str.lower() == util_name]
    # urdb_df_1 = urdb_df_1[urdb_df_1['name'].str.contains('tou|tod|time', case=False, na=False)]
    urdb_df_1 = urdb_df_1[
        urdb_df_1['name'].str.contains('tou|tod|time', case=False, na=False) &
        ~urdb_df_1['name'].str.contains('3-phase|3-tier|three phase', case=False, na=False)
    ]
    # Keep rows with no enddate
    filtered_urdb_df_1 = urdb_df_1[urdb_df_1['enddate'].isna()]
    # If no rows found, filter rows with enddate later than 2025
    # Repeat for 2024, 2022, 2021, and 2020
    for year in range(2025, 2019, -1):
        if filtered_urdb_df_1.empty:
            filtered_urdb_df_1 = urdb_df_1[pd.to_datetime(urdb_df_1['enddate'], errors='coerce') > pd.Timestamp(f'{year}-01-01')]
    filtered_urdb_df_2 = filtered_urdb_df_1[pd.to_datetime(filtered_urdb_df_1['latest_update'], errors='coerce') > pd.Timestamp(f'2025-01-01')]
    for year in range(2024, 2019, -1):
        if filtered_urdb_df_2.empty:
            filtered_urdb_df_2 = filtered_urdb_df_1[pd.to_datetime(filtered_urdb_df_1['latest_update'], errors='coerce') > pd.Timestamp(f'{year}-01-01')]

    utility_labels = filtered_urdb_df_1['label'].to_list()
    ResSales, ResRev, ResCount = ctdf1['ResSales'].unique()[0], ctdf1['ResRev'].unique()[0], ctdf1['ResCount'].unique()[0]
    if len(filtered_urdb_df_2) == 1:
        utility_label = filtered_urdb_df_2['label'].to_list()[0]
        urdb_name = filtered_urdb_df_2['name'].to_list()[0]
        import subprocess
        import os
        # utility_label = '643476222faee2f0f800d8b1'
        def run_julia_script(utility_label):
            
            julia_script = f"""
            using CSV
            using DataFrames

            include(raw\"""urdb.jl\""")
            include(raw\"""electric_tariff.jl\""")
            include(raw\"""utils.jl\""")
            using CSV
            # Load the CSV file
            function process_urdb_data(urdb_label, year=2025, time_steps_per_hour=1)
                usurdb_data = CSV.read(raw\"""C:\\\\Users\\\\jiahuic\\\\University of Michigan Dropbox\\\\Jiahui Chen\\\\PVEV\\\\julia\\\\filtered_usurdb_2023later.csv\""", DataFrame)

                # Find the 'utility' corresponding to the 'urdb_label'
                utility = usurdb_data[usurdb_data.label .== urdb_label, :utility][1]
                name = usurdb_data[usurdb_data.label .== urdb_label, :name][1]

                # Download and parse URDB data
                urdb_response = download_urdb(urdb_label)
                rate = URDBrate(urdb_response, year; time_steps_per_hour=time_steps_per_hour)

                # Extract energy_rates and fixed_monthly_charge
                energy_rates = vec(rate.energy_rates[:, 1])  # Ensure energy_rates is a 1D array
                fixed_monthly_charge = rate.fixed_monthly_charge
                # Determine if energy rates vary within the first 48 elements
                tou = any(diff(energy_rates[1:48]) .!= 0) ? '1' : '0'

                # Create a DataFrame
                df = DataFrame(energy_rates=energy_rates[:, 1], fixed_monthly_charge=fill(fixed_monthly_charge, length(energy_rates)), tou=fill(tou, length(energy_rates)))
                # Write to CSV with 'name' in the file name
                file_name = "C:\\\\Users\\\\jiahuic\\\\University of Michigan Dropbox\\\\Jiahui Chen\\\\PVEV\\\\julia\\\\Utility_structure\\\\$(urdb_label).csv"
                CSV.write(file_name, df)
            end
            process_urdb_data(raw\"""{utility_label}\""")


            """

            # Write the Julia script to a temporary file
            with open(f"{basedir}\\\\PVEV\\\\julia\\\\temp_script.jl", "w") as file:
                file.write(julia_script)

            # Call Julia to execute the script
            # julia_executable = r"E:\Program Files (x86)\Julia-1.11.4\bin\julia.exe"  # Full path to Julia executable
            julia_executable = r'C:\Users\jiahuic\AppData\Local\Microsoft\WindowsApps\julia.EXE'
            try:
                result = subprocess.run([julia_executable, f"{basedir}\\\\PVEV\\\\julia\\\\temp_script.jl"], capture_output=True, text=True, check=True)
                print("Julia Output:")
                print(result.stdout)
            except subprocess.CalledProcessError as e:
                print("Error running Julia script:")
                print(e.stderr)

            # Clean up the temporary file
            os.remove(f"{basedir}\PVEV\julia\\\\temp_script.jl")
        # run_julia_script(utility_label)
        run_julia_script(utility_label)
        rate = pd.read_csv(basedir + r'\\PVEV\julia\Utility_structure\\'+utility_label+'.csv')
        load_name = os.listdir(basedir + r'\\Ford_CC\ResStock\indi_noelec_fossilfuel\\'+str(tempregi)+'_0_0')
        load = pd.read_csv(basedir + r'\\Ford_CC\ResStock\indi_noelec_fossilfuel\\'+str(tempregi)+'_0_0\\'+load_name[0])
        rate_avg = (sum(rate['energy_rates']*load['0'])+rate['fixed_monthly_charge'][0]*12)/sum(load['0'])
        # Map the 'label' from filtered_urdb_df_1 to ctdf based on the 'matched_utility' column
        ctdf.loc[ctdf['tempreg'] == tempregi, 'urdb_label'] = utility_label
        ctdf.loc[ctdf['tempreg'] == tempregi, 'rate_avg'] = rate_avg
        ctdf.loc[ctdf['tempreg'] == tempregi, 'name'] = urdb_name

        # Save the modified DataFrame to a new file with tempregi in its name
        output_path = os.path.join(basedir, f'PVEV\\ctdfs\\tempreg_{tempregi}_updated.csv')
        ctdf.loc[ctdf['tempreg'] == tempregi].to_csv(output_path, index=False)

    else:
        non_single.append([tempregi])

In [ ]:
non_single

In [ ]:
np.where(non_single == [380])

manual correction in ctdf_rate_check
Kentucky Utilities Co: rate set by hand

In [ ]:
# no TOU, urdb_df_1 as handle
tempregi = non_single[265  ][0]
urdb_df = pd.read_csv(basedir + r'\\PVEV\julia\residential_usurdb.csv')
ctdf = pd.read_csv(basedir + r'\\PVEV\utility_matching\tempreg_withutility_usurdb_filled_sales_state.csv')
ctdf1 = ctdf.loc[ctdf['tempreg'] == tempregi]
util_name = ctdf1['matched_utility'].str.lower().unique()[0]
urdb_df_1 = urdb_df.loc[urdb_df['utility'].str.lower() == util_name]
urdb_df_2 = urdb_df_1[urdb_df_1['enddate'].isna()]
# urdb_df_2 = urdb_df_1
# Repeat for 2024, 2022, 2021, and 2020
for year in range(2025, 2019, -1):
    if urdb_df_2.empty:
        urdb_df_2 = urdb_df_1[pd.to_datetime(urdb_df_1['enddate'], errors='coerce') > pd.Timestamp(f'{year}-01-01')]
urdb_df_3 = urdb_df_2[pd.to_datetime(urdb_df_2['latest_update'], errors='coerce') > pd.Timestamp(f'2025-01-01')]
for year in range(2024, 2019, -1):
    if urdb_df_3.empty:
        urdb_df_3 = urdb_df_2[pd.to_datetime(urdb_df_2['latest_update'], errors='coerce') > pd.Timestamp(f'{year}-01-01')]
print(tempregi)
# Display partial contents of urdb_df_1 without text wrapping
pd.set_option('display.max_colwidth', 40)  # Disable text wrapping
utility_labels = urdb_df_3['label'].to_list()
urdb_df_3


In [ ]:
tempregi = non_single[265  ][0]
urdb_df = pd.read_csv(basedir + r'\\PVEV\julia\filtered_usurdb_2023later.csv')
ctdf = pd.read_csv(basedir + r'\\PVEV\utility_matching\tempreg_withutility_usurdb_filled_sales_state.csv')
ctdf1 = ctdf.loc[ctdf['tempreg'] == tempregi]
util_name = ctdf1['matched_utility'].str.lower().unique()[0]
urdb_df_1 = urdb_df.loc[urdb_df['utility'].str.lower() == util_name]
# urdb_df_1 = urdb_df_1[urdb_df_1['name'].str.contains('tou|tod|time', case=False, na=False)]
urdb_df_1 = urdb_df_1[
    urdb_df_1['name'].str.contains('tou|tod|time|EV|electric vehicle', case=False, na=False) &
    ~urdb_df_1['name'].str.contains('3-phase|3-tier|three phase', case=False, na=False)]
# Keep rows with no enddate
filtered_urdb_df_1 = urdb_df_1[urdb_df_1['enddate'].isna()]
# filtered_urdb_df_1 = urdb_df_1
# If no rows found, filter rows with enddate later than 2025
# Repeat for 2024, 2022, 2021, and 2020
for year in range(2025, 2019, -1):
    if filtered_urdb_df_1.empty:
        filtered_urdb_df_1 = urdb_df_1[pd.to_datetime(urdb_df_1['enddate'], errors='coerce') > pd.Timestamp(f'{year}-01-01')]
filtered_urdb_df_2 = filtered_urdb_df_1[pd.to_datetime(filtered_urdb_df_1['latest_update'], errors='coerce') > pd.Timestamp(f'2025-01-01')]
for year in range(2024, 2019, -1):
    if filtered_urdb_df_2.empty:
        filtered_urdb_df_2 = filtered_urdb_df_1[pd.to_datetime(filtered_urdb_df_1['latest_update'], errors='coerce') > pd.Timestamp(f'{year}-01-01')]
utility_labels = filtered_urdb_df_2['label'].to_list()
ResSales, ResRev, ResCount = ctdf1['ResSales'].unique()[0], ctdf1['ResRev'].unique()[0], ctdf1['ResCount'].unique()[0]
print(tempregi)
print(filtered_urdb_df_2['name'].to_list())
print(filtered_urdb_df_2['description'].to_list())

In [ ]:
filtered_urdb_df_2

In [ ]:
for inde in list(filtered_urdb_df_2.index):
    print(filtered_urdb_df_2['description'][inde])
    print(filtered_urdb_df_2['name'][inde])
    print(utility_label)
    utility_label = filtered_urdb_df_2['label'][inde]

In [ ]:
inde = 12871	
print(urdb_df_1['description'][inde])
print(urdb_df_1['source'][inde])
utility_label = urdb_df_1['label'][inde]

In [ ]:
import subprocess
import os
# utility_labels = ['5f2828485457a3511bb5c37f']
def run_julia_script(utility_label):
    
    julia_script = f"""
    using CSV
    using DataFrames

    include(raw\"""urdb.jl\""")
    include(raw\"""electric_tariff.jl\""")
    include(raw\"""utils.jl\""")
    using CSV
    # Load the CSV file
    function process_urdb_data(urdb_label, year=2025, time_steps_per_hour=1)
        usurdb_data = CSV.read(raw\"""C:\\\\Users\\\\jiahuic\\\\University of Michigan Dropbox\\\\Jiahui Chen\\\\PVEV\\\\julia\\\\filtered_usurdb_2023later.csv\""", DataFrame)

        # Find the 'utility' corresponding to the 'urdb_label'
        utility = usurdb_data[usurdb_data.label .== urdb_label, :utility][1]
        name = usurdb_data[usurdb_data.label .== urdb_label, :name][1]

        # Download and parse URDB data
        urdb_response = download_urdb(urdb_label)
        rate = URDBrate(urdb_response, year; time_steps_per_hour=time_steps_per_hour)

        # Extract energy_rates and fixed_monthly_charge
        energy_rates = vec(rate.energy_rates[:, 1])  # Ensure energy_rates is a 1D array
        fixed_monthly_charge = rate.fixed_monthly_charge
        # Determine if energy rates vary within the first 48 elements
        tou = any(diff(energy_rates[1:48]) .!= 0) ? '1' : '0'

        # Create a DataFrame
        df = DataFrame(energy_rates=energy_rates[:, 1], fixed_monthly_charge=fill(fixed_monthly_charge, length(energy_rates)), tou=fill(tou, length(energy_rates)))
        # Write to CSV with 'name' in the file name
        file_name = "C:\\\\Users\\\\jiahuic\\\\University of Michigan Dropbox\\\\Jiahui Chen\\\\PVEV\\\\julia\\\\Utility_structure\\\\$(urdb_label).csv"
        CSV.write(file_name, df)
    end
    process_urdb_data(raw\"""{utility_label}\""")


    """

    # Write the Julia script to a temporary file
    with open(f"{basedir}\\\\PVEV\\\\julia\\\\temp_script.jl", "w") as file:
        file.write(julia_script)

    # Call Julia to execute the script
    # julia_executable = r"E:\Program Files (x86)\Julia-1.11.4\bin\julia.exe"  # Full path to Julia executable
    julia_executable = r'C:\Users\jiahuic\AppData\Local\Microsoft\WindowsApps\julia.EXE'
    try:
        result = subprocess.run([julia_executable, f"{basedir}\\\\PVEV\\\\julia\\\\temp_script.jl"], capture_output=True, text=True, check=True)
        print("Julia Output:")
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print("Error running Julia script:")
        print(e.stderr)

    # Clean up the temporary file
    # os.remove(f"{basedir}\PVEV\julia\\\\temp_script.jl")
# run_julia_script(utility_label)
# for inde in range(12802, 12809):
# utility_label = urdb_df_1['label'][inde]
print(ctdf1['avg_rate'].to_list()[0])
for utility_label in utility_labels:
    run_julia_script(utility_label)
    rate = pd.read_csv(basedir + r'\\PVEV\julia\Utility_structure\\'+utility_label+'.csv')
    load_name = os.listdir(basedir + r'\\Ford_CC\ResStock\indi_noelec_fossilfuel\\'+str(tempregi)+'_0_0')
    load = pd.read_csv(basedir + r'\\Ford_CC\ResStock\indi_noelec_fossilfuel\\'+str(tempregi)+'_0_0\\'+load_name[0])
    rate_avg = (sum(rate['energy_rates']*load['0'])+rate['fixed_monthly_charge'][0]*12)/sum(load['0'])
    index = urdb_df[urdb_df['label'].str.lower() == utility_label.lower()].index[0]
    print(index,rate_avg)

In [ ]:
for inde in [12100]:
    print(ctdf1['avg_rate'].to_list()[0])
    utility_label = urdb_df_1['label'][inde]
    run_julia_script(utility_label)
    rate = pd.read_csv(basedir + r'\\PVEV\julia\Utility_structure\\'+utility_label+'.csv')
    load_name = os.listdir(basedir + r'\\Ford_CC\ResStock\indi_noelec_fossilfuel\\'+str(tempregi)+'_0_0')
    load = pd.read_csv(basedir + r'\\Ford_CC\ResStock\indi_noelec_fossilfuel\\'+str(tempregi)+'_0_0\\'+load_name[0])
    rate_avg = (sum(rate['energy_rates']*load['0'])+rate['fixed_monthly_charge'][0]*12)/sum(load['0'])
    print(utility_label,rate_avg)

In [ ]:
# tou, single RGU
urdb_name = filtered_urdb_df_2['name'][inde]
ctdf.loc[ctdf['tempreg'] == tempregi, 'urdb_label'] = utility_label
ctdf.loc[ctdf['tempreg'] == tempregi, 'rate_avg'] = rate_avg
ctdf.loc[ctdf['tempreg'] == tempregi, 'name'] = urdb_name

# Save the modified DataFrame to a new file with tempregi in its name
output_path = os.path.join(basedir, f'PVEV\\ctdfs\\tempreg_{tempregi}_updated.csv')
ctdf.loc[ctdf['tempreg'] == tempregi].to_csv(output_path, index=False)


In [ ]:
tempreglst = ctdf.loc[ctdf['matched_utility'].str.lower() == filtered_urdb_df_2['utility'].str.lower().unique()[0], 'tempreg'].unique()
print(tempreglst)

In [ ]:
# tou multiple RGUs
for tempregi in tempreglst:
# for tempregi in [423]:
    ctdf.loc[ctdf['tempreg'] == tempregi, 'urdb_label'] = utility_label
    ctdf.loc[ctdf['tempreg'] == tempregi, 'rate_avg'] = rate_avg
    ctdf.loc[ctdf['tempreg'] == tempregi, 'name'] = urdb_name

    # Save the modified DataFrame to a new file with tempregi in its name
    output_path = os.path.join(basedir, f'PVEV\\ctdfs\\tempreg_{tempregi}_updated.csv')
    ctdf.loc[ctdf['tempreg'] == tempregi].to_csv(output_path, index=False)


In [ ]:
# no tou single RGUs
urdb_df_1 = urdb_df_3
utility_label = urdb_df_1['label'][inde]
urdb_name = urdb_df_1['name'][inde]
ctdf.loc[ctdf['tempreg'] == tempregi, 'urdb_label'] = utility_label
ctdf.loc[ctdf['tempreg'] == tempregi, 'rate_avg'] = rate_avg
ctdf.loc[ctdf['tempreg'] == tempregi, 'name'] = urdb_name

# Save the modified DataFrame to a new file with tempregi in its name
output_path = os.path.join(basedir, f'PVEV\\ctdfs\\tempreg_{tempregi}_updated.csv')
ctdf.loc[ctdf['tempreg'] == tempregi].to_csv(output_path, index=False)


In [ ]:
# no tou multiple RGUs
# for tempregi in [383,384]:
# tempreglst = ctdf.loc[ctdf['matched_utility'].str.lower() == filtered_urdb_df_2['utility'].str.lower().unique()[0], 'tempreg'].unique()
# for tempregi in tempreglst:
for tempregi in [380,381,382]:
    utility_label = urdb_df_3['label'][inde]
    urdb_name = urdb_df_3['name'][inde]
    ctdf.loc[ctdf['tempreg'] == tempregi, 'urdb_label'] = utility_label
    ctdf.loc[ctdf['tempreg'] == tempregi, 'rate_avg'] = rate_avg
    ctdf.loc[ctdf['tempreg'] == tempregi, 'name'] = urdb_name

    # Save the modified DataFrame to a new file with tempregi in its name
    output_path = os.path.join(basedir, f'PVEV\\ctdfs\\tempreg_{tempregi}_updated.csv')
    ctdf.loc[ctdf['tempreg'] == tempregi].to_csv(output_path, index=False)


In [ ]:
print(inde,tempregi,rate_avg)
try:
    print(urdb_df_3['name'][inde])
except:print(filtered_urdb_df_2['name'][inde])

Alabama PEV TOU calculated manually

In [ ]:
rate = pd.read_csv(basedir + r'\\PVEV\julia\Utility_structure\649c4a551ed2da5c330a930e_missing.csv')
load_name = os.listdir(basedir + r'\\Ford_CC\ResStock\indi_noelec_fossilfuel\\'+str(tempregi)+'_0_0')
load = pd.read_csv(basedir + r'\\Ford_CC\ResStock\indi_noelec_fossilfuel\\'+str(tempregi)+'_0_0\\'+load_name[0])
rate_avg = (sum(rate['energy_rates']*load['0'])+rate['fixed_monthly_charge'][0]*12)/sum(load['0'])
print(rate_avg)


In [ ]:
utility_label = '649c4a551ed2da5c330a930e_missing'
urdb_name = 'Plug-In Electric Vehicle - Family Dwelling Service'

In [ ]:
tempreglst = ctdf.loc[ctdf['matched_utility'].str.lower() == 'alabama power co', 'tempreg'].unique()

In [ ]:
# tou multiple RGUs
for tempregi in tempreglst:
# for tempregi in range(348,351):
    ctdf.loc[ctdf['tempreg'] == tempregi, 'urdb_label'] = utility_label
    ctdf.loc[ctdf['tempreg'] == tempregi, 'rate_avg'] = rate_avg
    ctdf.loc[ctdf['tempreg'] == tempregi, 'name'] = urdb_name

    # Save the modified DataFrame to a new file with tempregi in its name
    output_path = os.path.join(basedir, f'PVEV\\ctdfs\\tempreg_{tempregi}_updated.csv')
    ctdf.loc[ctdf['tempreg'] == tempregi].to_csv(output_path, index=False)
